In [2]:
import numpy as np 
import pandas as pd 

import mlflow
import mlflow.sklearn
import dagshub

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import os 

/media/anni/B0BE7F0CBE7EC9FC/projects-with-mlops/mlops-mini-project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dagshub.init(repo_owner='anni0955', repo_name='mlops-mini-project', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/anni0955/mlops-mini-project.mlflow')

Accessing as anni0955

Initialized MLflow to track repo "anni0955/mlops-mini-project"

Repository anni0955/mlops-mini-project initialized!

In [4]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv')
df.head()

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,@dannycastillo We want to trade with someone w...


In [5]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


def lemmatization(text):
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return ' '.join(text)

def remove_stop_words(text):
    text = [word for word in str(text).split() if word not in stop_words]
    return ' '.join(text)

def remove_numbers(text):
    text = ''.join(char for char in text if not char.isdigit())
    return text

def lowercase(text):
    text = text.split()
    text = [word.lower() for word in text]
    return ' '.join(text)

def remove_punctuations(text):
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace(':', '')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    try:
        df['content'] = df['content'].apply(lowercase)
        df['content'] = df['content'].apply(remove_urls)
        df['content'] = df['content'].apply(remove_punctuations)
        df['content'] = df['content'].apply(remove_numbers)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'error in text_normalization: {e}')
        raise 

In [6]:
df = normalize_text(df)

In [7]:
x = df['sentiment'].isin(['happiness', 'sadness'])
df = df[x]

In [8]:
df['sentiment'] = df['sentiment'].replace({'sadness': 0, 'happiness': 1})
df.head()

/tmp/ipykernel_79197/2161997397.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'sadness': 0, 'happiness': 1})


,tweet_id,sentiment,content
1,1956967666,0,layin n bed headache ughhhh waitin call
2,1956967696,0,funeral ceremony gloomy friday
6,1956968487,0,sleep im thinking old friend want married damn...
8,1956969035,0,charviray charlene love miss
9,1956969172,0,kelcouch sorry least friday


In [9]:
mlflow.set_experiment('bow vs tfidf')

<Experiment: artifact_location='mlflow-artifacts:/25bd667e8a7a44baa7644fbae1d80170', creation_time=1774691661675, experiment_id='1', last_update_time=1774691661675, lifecycle_stage='active', name='bow vs tfidf', tags={}, workspace='default'>

In [10]:
vectorizers = {
    'bow': CountVectorizer(),
    'tfidf': TfidfVectorizer()
}

In [11]:
algorithms = {
    'lr': LogisticRegression(),
    'MultinomialNB': MultinomialNB(),
    'xgboost': XGBClassifier(),
    'rf': RandomForestClassifier(),
    'gb': GradientBoostingClassifier()
}

In [12]:
with mlflow.start_run(run_name='all experiments') as parent_run:
    for algo_name, algorithm in algorithms.items():
        for vec_name, vectorizer in vectorizers.items():
            with mlflow.start_run(run_name=f'{algo_name} with {vec_name}', nested=True) as child_run:
                x = vectorizer.fit_transform(df['content'])
                y = df['sentiment']

                x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.2, random_state=42)
                
                mlflow.log_param('vectorizer', vec_name)
                mlflow.log_param('algorithm', algo_name)
                mlflow.log_param('test_size', .2)

                model = algorithm
                model.fit(x_train, y_train)

                if algo_name == 'lr':
                    mlflow.log_param('C', model.C)

                elif algo_name == 'MultinomialNB':
                    mlflow.log_param('alpha', model.alpha)

                elif algo_name == 'xgboost':
                    mlflow.log_param('n_estimators', model.n_estimators)
                    mlflow.log_param('learning_rate', model.learning_rate)

                elif algo_name == 'rf':
                    mlflow.log_param('n_estimators', model.n_estimators)
                    mlflow.log_param('max_depth', model.max_depth)
                    
                elif algo_name == 'gb':
                    mlflow.log_param('n_estimators', model.n_estimators)
                    mlflow.log_param('max_depth', model.max_depth)
                    mlflow.log_param('learning_rate', model.learning_rate)

                y_pred = model.predict(x_test)
                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred)
                recall = recall_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred)

                mlflow.log_metric('accuracy', accuracy)
                mlflow.log_metric('precision', precision)
                mlflow.log_metric('recall', recall)
                mlflow.log_metric('f1', f1)

                mlflow.sklearn.log_model(model, 'model')
                
            
                notebook_path = 'exp2_bow_vs_tfidf.ipynb'
                os.system(f'jupyter nbconvert --to notebook --execute --inplace {notebook_path}')

                mlflow.log_artifact(str(notebook_path))

                
                print(f'Algorithm: {algo_name}, feature engineering: {vec_name}')
                print('accuracy:', accuracy)
                print('precision:', precision)
                print('recall:', recall)
                print('f1:', f1)


2026/03/28 15:39:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 15:39:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime d

Algorithm: lr, feature engineering: bow
accuracy: 0.7995180722891566
precision: 0.7882579403272377
recall: 0.8068965517241379
f1: 0.7974683544303798
🏃 View run lr with bow at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/1/runs/430b0c99a7d049b8a30231bbaa720f38
🧪 View experiment at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/1


2026/03/28 15:40:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 15:40:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime d

Algorithm: lr, feature engineering: tfidf
accuracy: 0.796144578313253
precision: 0.779245283018868
recall: 0.8137931034482758
f1: 0.796144578313253
🏃 View run lr with tfidf at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/1/runs/0e36321876b64ea4be70162ad30941cb
🧪 View experiment at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/1


2026/03/28 15:40:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 15:40:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime d

...

Algorithm: rf, feature engineering: tfidf
accuracy: 0.7677108433734939
precision: 0.765702891326022
recall: 0.7566502463054188
f1: 0.7611496531219029
🏃 View run rf with tfidf at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/1/runs/d14793886c7d4acda4f2530e9f662ef5
🧪 View experiment at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/1


2026/03/28 15:50:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 15:50:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime d

Algorithm: gb, feature engineering: bow
accuracy: 0.723855421686747
precision: 0.807799442896936
recall: 0.5714285714285714
f1: 0.6693594922100404
🏃 View run gb with bow at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/1/runs/31fff4ca11ca42929e7a88b8921437e0
🧪 View experiment at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/1


2026/03/28 15:51:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 15:51:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime d

Algorithm: gb, feature engineering: tfidf
accuracy: 0.7161445783132531
precision: 0.8042857142857143
recall: 0.554679802955665
f1: 0.6565597667638484
🏃 View run gb with tfidf at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/1/runs/78198959140d436282c3121510345a83
🧪 View experiment at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/1
🏃 View run all experiments at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/1/runs/220d3f37c82e42e0a21d4781867b996b
🧪 View experiment at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/1
